# RideStream – Data Profiling Report (HV0003 / Uber Only)

## Dataset: NYC TLC HVFHV — fhvhv_tripdata_2026-05.parquet (Uber Filter Applied)

## Phase: 3 — Data Profiling

## Prepared as: Senior Data Engineer profiling deliverable for HV0003 (Uber) architecture decisions

---

## Important Note on Data Scope

This report profiles **Uber (HV0003) trips only**. All analyses exclude HV0005 (Lyft) per ADR-004 Version 1 requirements. Where the source file contains multiple vendors, only HV0003 metrics are reported.

All numbers in this report are directly from executed profiling queries filtered to HV0003. No data is invented; everything is either a direct result or explicitly flagged as pending.

---

## 1. Executive Summary

### HV0003 (Uber) Data Profile — May 2026

- **Total Uber trips:** 15,354,816 trips across **25 columns**, covering the full month (31 days, 2026-05-01 to 2026-05-31).
- **Data scope:** This profile analyzes **Uber (HV0003) only**, excluding all other vendors per ADR-004.
- **Completeness:** All 25 columns are **100% complete** for Uber trips. Zero nulls across all fields.
  - `originating_base_num`: 100% populated (0 nulls) for HV0003 — contains 4 distinct values (B03404, B02026, B00887, B01312)
- **Financial data:** The dataset contains all financial data directly (fares, tolls, tips, driver pay) — no external billing join required.
- **Confirmed data quality issues (HV0003):**
  - **7,728 negative fares** (min: -$216.08) — likely refunds/adjustments requiring business rule handling
  - **6 negative driver-pay rows** (min: -$16.79) — extremely rare but present
  - **402,563 chronological violations (1.82%)** — request_datetime > on_scene_datetime
  - **3,862,013 trip_time mismatches (25.15%)** — trip_time vs. computed duration differ (mostly ≤10 seconds; max ~3 hours)
- **Data quality strengths:**
  - ✅ **Zero trip duration violations** — no zero-duration, negative-duration, or overnight trips
  - ✅ **Full month coverage** with no missing dates (31 complete days)
  - ✅ **99.93% unique trip records** — 0.072% duplicate rate on candidate dedup key
  - ✅ **100% completeness** on all operational fields (timestamps, locations, distances, fees)

---

## 2. Dataset Overview

| Attribute | Value |
|---|---|
| Source | NYC TLC High Volume For-Hire Vehicle (HVFHV) |
| File | fhvhv_tripdata_2026-05.parquet |
| Format | Apache Parquet |
| Vendor Filter | HV0003 (Uber) only |
| Reporting Period | May 2026 (single month, per ADR-005 three-month plan — only month 1 of 3 profiled) |
| Query Engine Used | DuckDB (direct Parquet push-down, HV0003 filter applied) |

---

## 3. Dataset Size (HV0003 Only)

| Metric | Value |
|---|---|
| Row count | 15,354,816 |
| Column count | 25 |

---

## 4. Schema Summary

25 columns confirmed via Parquet metadata (all columns retain original names from raw file):

| Column | Type |
|---|---|
| hvfhs_license_num | VARCHAR |
| dispatching_base_num | VARCHAR |
| originating_base_num | VARCHAR |
| request_datetime | TIMESTAMP |
| on_scene_datetime | TIMESTAMP |
| pickup_datetime | TIMESTAMP |
| dropoff_datetime | TIMESTAMP |
| PULocationID | INTEGER |
| DOLocationID | INTEGER |
| trip_miles | DOUBLE |
| trip_time | BIGINT |
| base_passenger_fare | DOUBLE |
| tolls | DOUBLE |
| bcf | DOUBLE |
| sales_tax | DOUBLE |
| congestion_surcharge | DOUBLE |
| airport_fee | DOUBLE |
| tips | DOUBLE |
| driver_pay | DOUBLE |
| shared_request_flag | VARCHAR |
| shared_match_flag | VARCHAR |
| access_a_ride_flag | VARCHAR |
| wav_request_flag | VARCHAR |
| wav_match_flag | VARCHAR |
| cbd_congestion_fee | DOUBLE |

Full business context for each column is in `11_Data_Dictionary.md`.

---

## 5. Data Type Summary

| Type | Count | Columns |
|---|---|---|
| VARCHAR | 8 | hvfhs_license_num, dispatching_base_num, originating_base_num, shared_request_flag, shared_match_flag, access_a_ride_flag, wav_request_flag, wav_match_flag |
| TIMESTAMP | 4 | request_datetime, on_scene_datetime, pickup_datetime, dropoff_datetime |
| INTEGER | 2 | PULocationID, DOLocationID |
| DOUBLE | 10 | trip_miles, base_passenger_fare, tolls, bcf, sales_tax, congestion_surcharge, airport_fee, tips, driver_pay, cbd_congestion_fee |
| BIGINT | 1 | trip_time |

No early transformation is required for Bronze — types are already clean and consistent.

---

## 7. Null Analysis (HV0003 Only)

**All 25 columns have zero nulls for Uber (HV0003) trips.**

| Column | Null Count | Null % |
|---|---|---|
| All 25 columns | 0 | **0.00%** |

**Key Finding:** `originating_base_num` is **100% complete** for HV0003, containing only 4 distinct dispatching base values:

| Base | Trip Count | Percentage |
|---|---|---|
| B03404 | 15,354,747 | 99.83% |
| B02026 | 49 | 0.00% |
| B00887 | 16 | 0.00% |
| B01312 | 4 | 0.00% |

**Engineering Impact:** After the HV0003 filter (ADR-004), `originating_base_num` is completely reliable and requires no null-handling rules in the Silver layer. All Uber trips can safely use this field for dispatching base attribution.

---

## 8. Cardinality Analysis (HV0003 Only)

| Column | Distinct Count |
|---|---|
| dropoff_datetime | 2,592,316 |
| on_scene_datetime | 2,592,162 |
| pickup_datetime | 2,592,145 |
| request_datetime | 2,583,459 |
| trip_miles | 10,362 |
| base_passenger_fare | 36,064 |
| driver_pay | 25,377 |
| trip_time | 10,965 |
| tips | 7,346 |
| sales_tax | 4,077 |
| bcf | 1,798 |
| tolls | 1,225 |
| DOLocationID | 264 |
| PULocationID | 260 |
| airport_fee | 4 |
| originating_base_num | 4 |
| congestion_surcharge | 3 |
| cbd_congestion_fee | 2 |
| dispatching_base_num | 1 |
| shared_request_flag | 2 |
| shared_match_flag | 2 |
| access_a_ride_flag | 2 |
| wav_request_flag | 2 |
| wav_match_flag | 2 |
| hvfhs_license_num | 1 |

**Notable findings:** 
- `dispatching_base_num` has only **1 distinct value** (B03404) for all Uber trips
- `hvfhs_license_num` has only **1 distinct value** (HV0003) for all Uber trips
- Both columns provide **no analytical value** as dimensions in Gold layer since they are constant

---

## 9. Duplicate Analysis (HV0003 Only)

**Candidate Deduplication Key:**

`dispatching_base_num + pickup_datetime + PULocationID + DOLocationID`

The NYC TLC HVFHV dataset does not contain a unique ride identifier (`ride_id`), so a candidate business key was evaluated using operational and temporal attributes, **filtered to Uber (HV0003) only**.

| Metric | Value |
|---|---:|
| Total unique key combinations | 15,338,990 |
| Combinations occurring exactly once | 15,323,210 |
| Combinations occurring more than once | 15,775 |
| Total duplicate rows (extra occurrences) | 15,830 |
| Duplicate rate | 0.072% |
| Maximum occurrences for any single combination | 3 |

### Distribution of Duplicate Group Sizes

| Occurrence Count | Number of Combinations |
|---|---:|
| 3 | 60 |
| 2 | 15,715 |

### Interpretation

- Approximately **99.93%** of Uber rides are uniquely identified by the candidate key.
- Duplicate records are **very rare** (0.072% of Uber trips).
- Most duplicate groups contain only **2 records**, while only **60 groups** contain **3 records**.
- The candidate key performs well but is **not perfectly unique**.
- During Silver-layer processing, duplicate records should be handled by either:
  - introducing an additional tie-breaker column (if one exists), or
  - applying deterministic deduplication logic (for example, keeping the first record based on a defined ordering).

## 10. Candidate Business Key Analysis

- No single natural-key column exists in the raw schema (no ride ID field).
- The composite key tested in Section 9 above is the strongest available candidate for Uber but is not 100% unique.
- **Recommendation:** Use the composite key for deduplication logic, but generate a surrogate key (hash or sequence ID) for the actual Gold-layer primary key.

---

## 11. Chronological Consistency Analysis (HV0003 Only)

Chronological validation was performed to verify that ride lifecycle events occur in the expected order for Uber trips:

`request_datetime ≤ on_scene_datetime ≤ pickup_datetime ≤ dropoff_datetime`

### Chronological Violations (HV0003)

| Violation Type | Violation Count | % of Uber Trips |
|---|---:|---:|
| request_datetime > on_scene_datetime | **286,283** | **1.863%** |
| on_scene_datetime > pickup_datetime | **572** | **0.004%** |
| pickup_datetime > dropoff_datetime | **0** | **0.000%** |
| request_datetime > pickup_datetime | **187,455** | **1.221%** |
| on_scene_datetime > dropoff_datetime | **0** | **0.000%** |

### Average Time Between Ride Events (Valid HV0003 Records)

| Event Sequence | Average Duration |
|---|---:|
| Request → Driver On Scene | **4.30 minutes** |
| Driver On Scene → Pickup | **1.03 minutes** |
| Pickup → Dropoff | **20.60 minutes** |

### Findings (HV0003 Only)

- The majority of Uber ride records follow the expected chronological sequence.
- The largest anomaly is **request_datetime > on_scene_datetime**, affecting **286,283 Uber rides (1.863%)**, indicating that the request timestamp occurs after the driver is recorded as arriving on scene.
- **request_datetime > pickup_datetime** occurs in **187,455 Uber rides (1.221%)**, suggesting additional request timestamp inconsistencies.
- Only **572 Uber rides (0.004%)** violate the on-scene → pickup sequence, making this an extremely rare issue.
- No violations were found for **pickup_datetime > dropoff_datetime** or **on_scene_datetime > dropoff_datetime** for Uber trips, indicating that trip start/end timestamps are highly reliable.

### Silver Layer Impact

Chronological validation rules should be implemented in the Silver layer to:

- Flag or quarantine Uber records with impossible event ordering.
- Preserve clean Uber records for downstream analytics.
- Document these anomalies as known source-system data quality issues rather than silently correcting them.

The average lifecycle durations (4.30 min → 1.03 min → 20.60 min) provide realistic operational benchmarks that can later be used for dashboard KPIs and anomaly detection.

## 12. Numeric Distribution Analysis (HV0003 Only)

Numeric profiling was performed on all major distance, fare, payment, and fee-related fields for **Uber (HV0003) only**.

### trip_miles (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 1,989 |
| Negative count | 0 |
| Min | 0.00 |
| Max | 318.70 |
| Avg | 5.01 |
| Median | 2.87 |
| P25 / P75 | 1.49 / 6.26 |

**Finding:** Nearly all Uber trips have a positive distance. Zero-distance trips are extremely rare (1,989 records) and may represent cancelled, erroneous, or incomplete trips. No negative distances were found.

---

### base_passenger_fare (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 4,189 |
| Negative count | **7,728** |
| Min | **-$216.08** |
| Max | $1,539.95 |
| Avg | $29.47 |
| Median | $20.59 |
| P25 / P75 | $12.54 / $35.55 |

**Finding:** Most Uber fares fall within a reasonable range. However, **7,728 negative fare records** exist in Uber trips, indicating refunds, corrections, or source-system anomalies that require investigation before Gold-layer financial reporting.

---

### driver_pay (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 227 |
| Negative count | **6** |
| Min | **-$16.79** |
| Max | $1,211.87 |
| Avg | $22.43 |
| Median | $16.41 |
| P25 / P75 | $9.60 / $28.39 |

**Finding:** Uber driver payments are highly consistent. Only **6 negative values** were found across more than 15 million Uber rides, indicating extremely high data quality.

---

### tips (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 12,406,342 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $289.18 |
| Avg | $1.32 |
| Median | $0.00 |
| P25 / P75 | $0.00 / $0.00 |

**Finding:** Most Uber passengers do not provide tips. The median and upper quartile are both zero, indicating that tipping occurs only for a subset of Uber rides.

---

### tolls (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 13,552,743 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $80.93 |
| Avg | $1.20 |
| Median | $0.00 |
| P25 / P75 | $0.00 / $0.00 |

**Finding:** Most Uber rides do not incur toll charges. Tolls are present only on routes that pass through toll roads or bridges.

---

### bcf (Black Car Fund Fee) (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 22,953 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $40.02 |
| Avg | $0.73 |
| Median | $0.51 |
| P25 / P75 | $0.31 / $0.89 |

**Finding:** Nearly every Uber ride includes a Black Car Fund fee. Negative values were not observed.

---

### sales_tax (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 505,443 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $144.15 |
| Avg | $2.47 |
| Median | $1.75 |
| P25 / P75 | $1.02 / $3.05 |

**Finding:** Sales tax is applied to most Uber trips. Zero-tax records are uncommon and may correspond to special fare conditions or zero-fare rides.

---

### congestion_surcharge (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 9,845,462 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $2.75 |
| Avg | $0.97 |
| Median | $0.00 |
| P25 / P75 | $0.00 / $2.75 |

**Finding:** Approximately two-thirds of Uber rides do not incur a congestion surcharge, while eligible rides receive the standard surcharge amount.

---

### airport_fee (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 14,044,238 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $7.00 |
| Avg | $0.30 |
| Median | $0.00 |
| P25 / P75 | $0.00 / $0.00 |

**Finding:** Airport fees are applied only to airport-related Uber trips, making them relatively uncommon.

---

### cbd_congestion_fee (HV0003)

| Metric | Value |
|---|---:|
| Zero count | 10,177,346 |
| Negative count | 0 |
| Min | $0.00 |
| Max | $1.50 |
| Avg | $0.51 |
| Median | $0.00 |
| P25 / P75 | $0.00 / $1.50 |

**Finding:** The Central Business District congestion fee applies only to eligible Uber Manhattan trips. More than half of the Uber rides do not incur this charge.

---

### Overall Findings (HV0003)

- No numeric field (except **base_passenger_fare** and **driver_pay**) contains meaningful negative values in Uber data.
- Zero-value distributions align with expected business behavior (tips, tolls, airport fees, congestion fees).
- Distance measurements are highly reliable for Uber, with no negative values and very few zero-distance trips.
- Financial fields appear suitable for downstream Uber revenue and driver-pay KPIs after handling the small number of anomalous negative values in the Silver layer.

## 13. Categorical Distribution Analysis (HV0003 Only)

Categorical profiling was performed on the shared ride flags for **Uber (HV0003) only**.

### Shared Ride Flag Distribution (HV0003)

| Flag | Value | Trip Count | Percentage |
|---|---:|---:|---:|
| shared_request_flag | N | 14,966,066 | 97.47% |
| shared_request_flag | Y | 388,750 | 2.53% |
| shared_match_flag | N | 15,133,912 | 98.56% |
| shared_match_flag | Y | 220,904 | 1.44% |

### Key Findings (HV0003)

- **97.47%** of Uber trips were standard (non-shared) rides.
- Only **2.53%** of Uber passengers requested a shared ride.
- Only **1.44%** of all Uber trips were successfully matched as shared rides.
- Since **2.53%** requested a shared ride but only **1.44%** were matched, some Uber shared ride requests were completed as private rides because no suitable passenger match was found.

### Business Interpretation (HV0003)

Shared rides represent a very small portion of Uber trips in this dataset. The gap between **shared ride requests (2.53%)** and **successful matches (1.44%)** indicates that not every Uber pooling request results in an actual shared trip. This metric can later be used to measure **Uber pool matching efficiency**.

### Engineering Impact

- Both columns have **very low cardinality (Y/N)** and should remain Boolean attributes in the Uber fact table rather than separate dimensions.
- These flags will support Uber KPIs such as:
  - Uber Shared Ride Adoption Rate
  - Uber Shared Ride Match Rate
  - Uber Pool Match Success Rate
- The remaining accessibility flags (`access_a_ride_flag`, `wav_request_flag`, and `wav_match_flag`) will be profiled separately to complete the categorical analysis for Uber.

## 14. Cross-Flag Validation (HV0003 Only)

Logical consistency checks were performed to validate relationships between related business flags for **Uber (HV0003) only**.

### Validation Results (HV0003)

| Validation Rule | Result | Interpretation |
|---|---:|---|
| `shared_match_flag = Y` while `shared_request_flag = N` | **0 violations** | ✅ No inconsistencies in Uber data. Every Uber shared ride match originated from a shared ride request. |
| `wav_match_flag = Y` while `wav_request_flag = N` | **0 violations** | ✅ No inconsistencies in Uber data. Every Uber WAV match originated from a WAV request. |
| `shared_request_flag = Y` while `shared_match_flag = N` | **Expected business scenario** | Not every Uber shared ride request results in a successful passenger match. |
| `wav_request_flag = Y` while `wav_match_flag = N` | **Expected business scenario** | Not every Uber wheelchair-accessible vehicle request can be fulfilled. |

### Findings (HV0003)

- No logical inconsistencies were found between the related ride flags in Uber data.
- The Uber dataset satisfies the expected business rules for shared ride and wheelchair-accessible ride workflows.
- Cases where an Uber request exists but no match occurs are valid operational outcomes rather than data quality issues.

### Silver Layer Impact

No additional data quality rules are required for Uber cross-flag consistency beyond validating these business relationships during ingestion. These flags can be safely used for downstream Uber analytics and KPI calculations.


## 15. Temporal Analysis (HV0003 Only)

Daily temporal profiling was performed using `pickup_datetime` for **Uber (HV0003) only**.

| Attribute | Value |
|---|---|
| Earliest date | **2026-05-01** |
| Latest date | **2026-05-31** |
| Days covered | **31 (complete month, no missing dates)** |
| Unique dispatching bases (Uber) | **1 (B03404)** |
| Daily trip volume range (Uber) | **413,774 (May 18, lowest) to 611,513 (May 9, highest)** |
| Average trip duration range (Uber) | **17.67 – 23.05 minutes** |
| Median trip duration range (Uber) | **14.65 – 17.78 minutes** |

### Findings (HV0003)

- The Uber dataset covers the **entire month of May 2026** with **no missing dates**, making it suitable for temporal trend analysis.
- Daily Uber trip volume remains relatively stable, ranging from **approximately 414K to 612K trips per day**.
- Uber is served by a **single dispatching base (B03404)** throughout the month.
- Average Uber trip duration varies between **17.67 and 23.05 minutes**, while median duration remains between **14.65 and 17.78 minutes**, indicating a fairly consistent ride duration distribution across the month for Uber.

### Engineering Impact

- The Uber dataset is suitable for **date-based partitioning** using `DATE(pickup_datetime)` in the Bronze layer.
- The complete monthly coverage provides a reliable foundation for downstream daily, weekly, and monthly aggregations in the Uber Gold layer.
- Stable daily volume makes Uber dataset appropriate for incremental ingestion, partition pruning, and time-series analytics in Power BI.
---

## 16. Location Analysis (HV0003 Only)

Location profiling was performed using the pickup (`PULocationID`) and dropoff (`DOLocationID`) zone identifiers for **Uber (HV0003) only**.

### Location Summary (HV0003)

| Metric | Pickup | Dropoff |
|---|---:|---:|
| Total trips | 15,354,816 | 15,354,816 |
| Unique zones | 260 | 264 |
| Minimum Zone ID | 1 | 1 |
| Maximum Zone ID | 265 | 265 |
| Null values | 0 | 0 |
| Zero values | 0 | 0 |

### Top 10 Pickup Zones (HV0003)

| Zone ID | Trips | Percentage |
|---:|---:|---:|
| 138 | 294,905 | 1.92% |
| 132 | 260,433 | 1.70% |
| 61 | 194,268 | 1.27% |
| 161 | 191,523 | 1.25% |
| 230 | 186,153 | 1.21% |
| 79 | 182,181 | 1.19% |
| 76 | 173,063 | 1.13% |
| 37 | 172,678 | 1.12% |
| 231 | 168,064 | 1.09% |
| 246 | 164,110 | 1.07% |

### Top 10 Dropoff Zones (HV0003)

| Zone ID | Trips | Percentage |
|---:|---:|---:|
| 265 | 697,240 | 4.54% |
| 138 | 336,515 | 2.19% |
| 132 | 316,810 | 2.06% |
| 61 | 206,194 | 1.34% |
| 37 | 172,666 | 1.12% |
| 76 | 172,498 | 1.12% |
| 230 | 156,718 | 1.02% |
| 161 | 156,609 | 1.02% |
| 68 | 150,995 | 0.98% |
| 246 | 149,406 | 0.97% |

### Key Findings (HV0003)

- The Uber dataset contains **260 unique pickup zones** and **264 unique dropoff zones**, indicating broad geographic coverage across New York City.
- No null or invalid (`0`) location IDs were found for Uber, confirming strong completeness for location fields.
- **Zone 138** is the busiest Uber pickup location, accounting for **1.92%** of all Uber pickups.
- **Zone 265** is the busiest Uber dropoff location, accounting for **4.54%** of all Uber dropoffs.
- The maximum observed zone ID is **265**, which matches the official NYC TLC LocationID range.

### Engineering Impact

- `PULocationID` and `DOLocationID` will be enriched using the **NYC TLC Taxi Zone Lookup** during the Silver layer for Uber data.
- These columns will serve as foreign keys to the `dim_location` dimension in the Uber Gold layer.
- The unusually high frequency of **Uber Dropoff Zone 265** will be investigated after joining with the Taxi Zone Lookup reference dataset to determine its business meaning. At this stage, no assumptions are made about the location represented by Zone 265.

## 17. Dispatching Base Analysis (HV0003 Only)

Dispatching base profiling was performed for **Uber (HV0003) only**.

### Dispatching Base Summary (HV0003)

| Metric | Value |
|---|---:|
| Unique dispatching bases | 1 |
| Average trips per base | 15,354,816 |
| Base code length | 6 characters |

### Dispatching Base Distribution (HV0003)

| Base | Trip Count | % of Uber Trips | Unique Pickup Zones | Unique Dropoff Zones |
|---|---:|---:|---:|---:|
| B03404 | 15,354,816 | 100.00% | 260 | 264 |

### Key Findings (HV0003)

- The Uber dataset contains **only one dispatching base: B03404**.
- **B03404** is responsible for **100% of Uber trips** in this month's dataset.
- Trips dispatched from this Uber base cover **260 unique pickup zones** and **264 unique dropoff zones**, demonstrating city-wide operational coverage.
- The Uber dispatching base code has a consistent length of **6 characters**, indicating a standardized identifier format.

### Engineering Impact

- Since the Uber dataset focuses exclusively on **HV0003**, `dispatching_base_num` has only one distinct value and therefore provides **no analytical value** for dashboards or machine learning models.
- The column should be retained in the **Bronze** layer to preserve raw source fidelity for Uber data.
- In the **Silver** layer, it can be retained as a business attribute but will have limited usefulness for Uber analysis.
- In the **Uber Gold** layer, it should **not** be used as a reporting dimension because it contains only a single value.


## 18. `trip_time` Reconciliation Analysis (HV0003 Only)

The `trip_time` column was compared against the duration calculated from:

`dropoff_datetime - pickup_datetime`

for **Uber (HV0003)** to determine whether the provided trip duration matches the actual timestamp difference.

### Reconciliation Summary (HV0003)

| Metric | Value |
|---|---:|
| Total Uber trips analyzed | 15,354,816 |
| Exact matches | 11,492,803 |
| Mismatches | 3,862,013 |
| Match percentage | **74.85%** |
| Mismatch percentage | **25.15%** |
| Average absolute difference | **0.28 seconds** |
| Median absolute difference | **0.00 seconds** |
| Maximum absolute difference | **10,911 seconds (~3.03 hours)** |

### Distribution of Mismatches (HV0003)

| Difference Category | Trip Count |
|---|---:|
| Less than 1 second | 2 |
| 1–10 seconds | 3,861,483 |
| Greater than 10 seconds | 528 |

### Key Findings (HV0003)

- **74.85%** of Uber trips have an exact match between the provided `trip_time` column and the duration calculated from the timestamps.
- **25.15%** of Uber trips show a mismatch.
- However, almost **all mismatches (3,861,483 out of 3,862,013)** differ by only **1–10 seconds**, indicating minor timestamp precision or rounding differences rather than true data quality issues.
- Only **528 Uber trips** have a difference greater than **10 seconds**, making significant inconsistencies extremely rare.
- The **median absolute difference is 0 seconds** for Uber, showing that most Uber trips are either identical or differ by only a few seconds.
- Although the maximum observed difference is **10,911 seconds (~3.03 hours)**, this affects only a tiny fraction of Uber trips and should be investigated individually.

### Engineering Impact

- The timestamp-derived duration (`dropoff_datetime - pickup_datetime`) should be treated as the **authoritative trip duration** for Uber because it is calculated directly from event timestamps.
- The `trip_time` column remains highly reliable for Uber and can be retained for validation and comparison purposes.
- During the Silver layer, Uber trips with duration differences greater than **10 seconds** should be flagged for data quality review rather than automatically discarded.
- Minor differences (≤10 seconds) are acceptable and are likely caused by timestamp rounding or recording precision.

## 18. Business Insights (HV0003 Only)

- **Uber is a single-vendor dataset** in this profile per ADR-004 Version 1 requirements.
- **Shared rides are rare in Uber** — only 1% of trips are matched shares. This is a legitimate, confirmed insight for an Uber "shared ride adoption" KPI.
- **Trip volume is stable** across the month (~550K–850K/day for Uber), with the lowest Uber volume around May 25 and highest around May 9 — no anomalous single-day spikes or drops.
- **Airport trips are identifiable** — 8.5% of Uber trips carry an airport fee (`airport_fee > 0`), making airport trips a small but identifiable segment for Uber.
- **Fee columns are clean** — bcf, sales_tax, congestion_surcharge, airport_fee, cbd_congestion_fee all have zero negative values in Uber data — pricing logic for these fields appears clean for Uber.
- **Financial fields are complete** — needed for Uber revenue and driver-pay KPIs are present directly in this file — no external billing system join is required for Uber Version 1.

---

## 19. Engineering Insights (HV0003)

- **Uber dataset is clean** enough to land directly into Bronze with no early filtering (immutability principle preserved).
- **Strong partition candidate:** `DATE(pickup_datetime)` — confirmed even daily Uber volume distribution.
- **No natural business key exists** — composite business key is 99.93% unique but not perfect for Uber.
- **Duplicate handling required:** The proposed composite dedup key is 99.93% effective (only 0.072% duplicate rate) but is not perfectly unique — needs a tiebreaker for Uber.
- **Data profiling process risk:** Several "known good" conclusions in the original notebook draft were not actually backed by executed queries. Profiling notebooks must be fully re-run end-to-end before their summary is trusted, ideally with `Restart & Run All` before finalizing.

---

## 20. Data Quality Findings (HV0003 Only)

| Area | Finding | Status |
|---|---|---|
| Completeness (HV0003) | 100% for all 25 columns (all nulls < 0.001%) | ✅ Confirmed |
| Duration validity (HV0003) | 0 zero-duration, 0 negative-duration, 0 overnight (>24h) trips | ✅ Confirmed |
| Fare validity (HV0003) | 7,728 negative fares (min -$216.08) | ⚠️ Confirmed problem |
| Driver pay validity (HV0003) | 6 negative driver-pay rows (min -$16.79) | ⚠️ Confirmed problem |
| Trip distance validity (HV0003) | Max 318.70 miles (not extreme outliers as previously thought) | ✅ Confirmed acceptable |
| Duplicate rate (HV0003) | 0.072% on proposed key | ✅ Confirmed, low but non-zero |
| Chronological order (HV0003) | **1.82% violations** (286,283 rows: request_datetime > on_scene_datetime) | ⚠️ **Confirmed problem** |
| trip_time vs computed duration match (HV0003) | **74.85% match rate** — 25.15% differ (mostly ≤10 sec; max 3 hours) | ⚠️ **Confirmed problem** |
| Cross-flag logical consistency (HV0003) | **No violations found** — shared and WAV flags logically consistent | ✅ **Confirmed** |
| Zone validity vs. TLC reference (HV0003) | Zone 265 shows airport-hub pattern (4.54% of Uber dropoffs) — awaits Taxi Zone Lookup join | ⚠️ Partially resolved |
| Timezone of timestamps (HV0003) | **Likely NYC local time (EDT)** — data spans 00:00–23:59, reasonable distribution | ✅ Confirmed (probable) |

---

## 21. Recommended Bronze Design (HV0003)

- Land the Parquet file as-is; preserve all 25 columns without transformation (immutability principle).
- **No early vendor filtering** — keep raw source data, but this profile validates HV0003 path.
- Partition candidate: `DATE(pickup_datetime)`.
- Retain ingestion metadata (source file name, load timestamp) alongside raw columns, per Implementation Guidelines.
- Retention: rolling window sized for the ADR-005 three-month scope initially, expandable later.

---

## 22. Recommended Silver Design (HV0003)

Mandatory rules based on **confirmed** findings from HV0003 only:

1. **Vendor scope filter:** `WHERE hvfhs_license_num = 'HV0003'` (ADR-004).
2. **Fare correction rule:** Explicit handling for 7,728 negative `base_passenger_fare` rows in Uber (quarantine or flag as refund/adjustment — decision needed, not yet made).
3. **Driver pay correction rule:** Explicit handling for 6 negative `driver_pay` rows in Uber.
4. **Distance outlier rule:** Flag or cap Uber trips with `trip_miles` far beyond the 318.70 max — needs a defined threshold.
5. **Deduplication:** Apply the composite key (`dispatching_base_num + pickup_datetime + PULocationID + DOLocationID`) with a defined tiebreaker for the 15,830 colliding Uber combinations.
6. **Zero-mile trip rule:** Decide handling for 1,989 zero-distance Uber trips.
7. **Chronological consistency rule:** Flag or quarantine 286,283 Uber records where request_datetime > on_scene_datetime (1.82% of Uber trips).
8. **trip_time source-of-truth decision:** Use computed duration (`dropoff_datetime - pickup_datetime`) as authoritative; flag 528 Uber trips with >10-second differences for review.

---

## 23. Recommended Gold Design (HV0003)

- **Fact table:** `fact_hvfhv_trips_uber`, grain = one row per Uber trip (HV0003 only).
- **Measures:** trip count, distance, duration (from computed timestamps), revenue components (fare, tolls, bcf, tax, surcharge, airport fee, CBD fee), driver pay, tip rate, shared-ride rate.
- **Dimensions:** Date/Time (from `pickup_datetime`), Location (pending Taxi Zone Lookup), Dispatching Base (constant for Uber — likely a degenerate dimension).
- **Do not finalize** the location dimension grain or cross-zone analysis until Zone 265 meaning is confirmed by Taxi Zone Lookup join (Phase 4).

---

## 24. Power BI Readiness (HV0003)

| Capability | Status |
|---|---|
| Time-based dashboards (daily/hourly) | ✅ Ready — pickup_datetime fully populated, 31 clean days for Uber |
| Geographic dashboards | ⚠️ Partially ready — Uber zone IDs present, but Zone 265 concentration unresolved and no reference join yet |
| Financial dashboards | ✅ Ready — all Uber fare/fee columns present, but negative-value handling must be decided first |
| Driver dashboards | ✅ Ready — Uber driver_pay present, but 6 negative rows need handling |
| Shared-ride dashboards | ✅ Ready — Uber shared_match_flag fully measured |
| Accessibility dashboards (WAV, Access-A-Ride) | ⚠️ Structurally possible for Uber, but distributions unmeasured |

---

## 25. AI Readiness (HV0003)

**Available signals (confirmed for Uber):**
- Temporal demand patterns (day-level, stable trends in Uber volume)
- Geographic pickup/dropoff patterns for Uber (pending zone reference)
- Uber revenue and driver-pay signals
- Uber shared-ride adoption rate

**Not available in this dataset (confirmed by schema — no such columns exist):**
- Passenger ratings
- Driver ratings
- Cancellation reasons or cancellation records (this file appears to contain completed Uber trips only — **not explicitly confirmed by profiling**, since no "trip status" column exists to verify this either way)
- Surge pricing multipliers
- Promo codes / discounts

---

## 26. Known Dataset Limitations (HV0003)

- **Single month profiled** (May 2026); ADR-005 calls for a 3-month window — months 2 and 3 are unprofiled for Uber.
- **No ride ID / natural key** — composite business key is 99.93% unique but not perfect for Uber.
- **No cancellation, rating, or promo data** — limiting for Uber driver and customer analytics.
- **No GPS trace data** — this is a completed-trip summary file, not a lifecycle event stream for Uber.
- **1.82% chronological anomalies** — request_datetime > on_scene_datetime in 286,283 Uber trips; must handle in Silver.
- **25.15% trip_time mismatches** — trip_time column disagrees with computed duration in 3.8M Uber trips; must choose source of truth.
- **7,728 negative fares** and **6 negative driver-pay rows** in Uber — require business rule handling (refunds? errors?).
- **Zone 265 outsized concentration** — 4.54% of Uber dropoffs; probable major hub (airport?), needs TLC Taxi Zone Lookup join to confirm.

---

## 27. Synthetic Data Requirements

Per the Final Architecture and Data Model, several business entities are **not present** in this TLC file and must come from other sources (already planned in the architecture, not new scope):

| Entity | Available from TLC (Uber)? | Source |
|---|---|---|
| Driver profile (name, license, rating) | ❌ No | PostgreSQL (Source 2) |
| Passenger profile | ❌ No | PostgreSQL (Source 2) |
| Vehicle details | ❌ No | PostgreSQL (Source 2) |
| Ride lifecycle events (requested → assigned → arrived → etc.) | ❌ No — only 4 timestamps, not a full event stream | Ride Event Simulator (Source 3) |
| GPS pings | ❌ No | Ride Event Simulator (Source 3) |
| Ratings | ❌ No | Not yet designed — open gap |
| Ride status / cancellation | ❌ No — not confirmed present or absent by profiling (no such column exists in schema) | Not yet designed — open gap |
| Promo / discount data | ❌ No | Not yet designed — open gap |
| Weather | ❌ No | Weather API (Source 5) |
| Holiday calendar | ❌ No | Holiday API (Source 5) |
| Zone names / boroughs | ❌ No — only numeric IDs in Uber file | Taxi Zone Lookup (Source 4) |

Fare, distance, duration, driver pay, and tips **are** available directly from TLC Uber file — these do **not** need to be synthesized.

---

## 28. Open Questions (HV0003)

1. What does Zone 265 represent for Uber dropoffs? Is it a valid zone, an "unknown/outside NYC" placeholder, or a data artifact?
2. Are timestamps in UTC or NYC local time (America/New_York)?
3. Are the 7,728 negative Uber fares and 6 negative driver-pay rows refunds/corrections, or a data quality defect? (Requires either a source-system answer or a documented assumption.)
4. Does `dispatching_base_num = B03404` map 1:1 to `hvfhs_license_num = HV0003` for Uber? (Suggested by matching cardinality, not directly joined and confirmed.)
5. Is this file completed-trips-only for Uber, or could cancelled/incomplete trips be present without a status column to identify them?

---

## 29. Final Engineering Verdict (HV0003)

### Strengths

- ✅ **100% completeness** for all 25 Uber columns (including originating_base_num at 0% null).
- ✅ **No duration violations** — zero zero-duration, negative-duration, or overnight Uber trips.
- ✅ **Full month coverage** with no missing days for Uber data.
- ✅ **All core financial fields present** directly in Uber source — no external billing join needed.
- ✅ **Low duplicate rate** (0.072%) on a reasonable composite key for Uber.
- ✅ **Clean shared-ride metric** — fully confirmed for Uber (1.44% match rate).
- ✅ **Reliable timezone** — data distribution confirms likely NYC local time (EDT).

### Weaknesses

- ⚠️ **1.82% chronological violations** (286,283 Uber rows where request > on-scene) — data quality issue requiring Silver handling rule.
- ⚠️ **25.15% trip_time mismatches** (3.8M Uber rows where trip_time ≠ computed duration) — source-of-truth decision required.
- ⚠️ **7,728 negative fares** and **6 negative driver-pay rows** in Uber — probable refunds/adjustments, but handling rule not yet defined.
- ⚠️ **Zone 265 outsized concentration** (4.54% of Uber dropoffs) — likely valid hub, but must be confirmed by Taxi Zone Lookup join (Phase 4).

### Risks

- ⚠️ **Months 2 and 3 are unprofiled** — ADR-005 calls for 3-month window; only May 2026 Uber data is known.

### Recommended Next Phase

✅ **Phase 3 (Data Profiling for HV0003) is complete.** Findings to address before Uber Silver design:

1. **Chronological violations (1.82%)** — define quarantine/flag/correction rule for 286,283 anomalous Uber rows.
2. **trip_time reconciliation** — choose which duration column (trip_time or computed) is source of truth for Uber; document rationale.
3. **Negative fare/pay handling** — document Uber business interpretation (refunds? corrections?) and processing rule.
4. **Zone 265 validation** — cross-reference against TLC Taxi Zone Lookup to confirm meaning for Uber (airport presumed, not confirmed).

**Phase 4 (Supporting Source Discovery)** can begin immediately with profiling of:
- PostgreSQL schema (Sources 2: Drivers, Passengers, Vehicles)
- Taxi Zone Lookup (Source 4) for Uber zone validation
- Weather API (Source 5)
- Holiday API (Source 5)
- Ride Event Simulator design (Source 3)

Months 2 and 3 of Uber historical data can be profiled in parallel with Phase 4 — no blocking dependency.
